In [9]:
# ==========================================
# PRAKTIKUM BAB 13: CNN & TRANSFER LEARNING
# ==========================================

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
import matplotlib.pyplot as plt

# Load dataset CIFAR-10 dan ambil subset kecil untuk efisiensi praktikum
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Normalisasi pixel ke rentang [0, 1]
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Ambil subset kecil (misal 2000 data train, 500 data test) agar cepat dieksekusi
X_train_sub, y_train_sub = X_train[:2000], y_train[:2000]
X_test_sub, y_test_sub = X_test[:500], y_test[:500]


print("=== PRAKTIKUM 1: CNN SEDERHANA DARI NOL ===")
# Sesuai kode pada modul (Praktikum 13.7)
model_cnn = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Latih CNN dari Nol
history_cnn = model_cnn.fit(
    X_train_sub, y_train_sub,
    validation_data=(X_test_sub, y_test_sub),
    epochs=5,
    batch_size=32,
    verbose=1
)


print("\n=== PRAKTIKUM 2: TRANSFER LEARNING DENGAN MOBILENETV2 ===")
# Load base model MobileNetV2 pre-trained ImageNet (tanpa layer fully connected atas)
base_model = MobileNetV2(input_shape=(32, 32, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # Bekukan bobot (feature extraction)

# Rangkai model transfer learning
model_tl = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax")
])

model_tl.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Latih Model Transfer Learning
history_tl = model_tl.fit(
    X_train_sub, y_train_sub,
    validation_data=(X_test_sub, y_test_sub),
    epochs=5,
    batch_size=32,
    verbose=1
)

# Visualisasi Kurva Perbandingan Akurasi
plt.figure(figsize=(8, 5))
plt.plot(history_cnn.history['val_accuracy'], label='CNN dari Nol (Val Acc)', marker='o')
plt.plot(history_tl.history['val_accuracy'], label='Transfer Learning (Val Acc)', marker='s')
plt.title("Perbandingan Akurasi Validasi: CNN Dari Nol vs Transfer Learning")
plt.xlabel("Epoch")
plt.ylabel("Akurasi")
plt.legend()
plt.show()

   557056/170498071 ━━━━━━━━━━━━━━━━━━━━ 21:11 7us/step

KeyboardInterrupt: 